**Model Evaluation**

The cell below takes `DATA_DIR` as input, pointing to the *partitioned_data* folder containing *4-shot.csv*, *8-shot.csv*, and *eval.csv*; and `RENDER_DIR` pointing to a folder of renders with names corresponding to *file_name* in the csvs. For each of the three closed-weight models evaluated, the cell will evaluate the model across all three setups and create a separate csv containing results of each model, for each setup in `EVAL_DIR` (so there would exist a *sonnet-4-shot.csv*, *gpt-baseline.csv*)

In [ ]:
DATA_DIR = "/scratch/gpfs/FELLBAUM/af3158/cos351/closed_weight_n_shot/partitioned_data"
RENDER_DIR = "/scratch/gpfs/FELLBAUM/af3158/cos351/mcd_dataset/renders"
EVAL_DIR = "/scratch/gpfs/FELLBAUM/af3158/cos351/closed_weight_n_shot/evals"

In [ ]:
TASK_DESCRIPTION = (
    "Respond with a single integer from 0 to 100. Do not write anything else.\n\n"
    "A Machine Gun Conversion Device (MCD) is a part that modifies a semi-automatic "
    "firearm to fire automatically. You will be shown a 3D model render accompanied by "
    "structured metadata. When present, items labelled \"role\": \"example\" are annotated "
    "reference cases with a known \"label\" (1 = MCD, 0 = not MCD). The item labelled "
    "\"role\": \"eval\" is the item to classify.\n\n"
    "Estimate the probability that the eval item is an MCD. "
    "0 = almost certainly not an MCD. 100 = almost certainly an MCD. 50 = maximum uncertainty.\n"
    "Integer:"
)

MODELS = {
    "sonnet": {
        "provider": "anthropic",
        "model_id": "claude-sonnet-4-6",
    },
    "gpt": {
        "provider": "openai",
        "model_id": "gpt-5.4",
    },
    "gemini": {
        "provider": "google",
        "model_id": "gemini-3.1-pro-preview",
    },
}

SETUPS = {
    "baseline": None,
    "4-shot": "4-shot.csv",
    "8-shot": "8-shot.csv",
}

In [ ]:
import os
import base64
import json
import re
import time
import pandas as pd
import anthropic
import openai
from google import genai
from google.genai import types
from PIL import Image
import io

anthropic_client = anthropic.Anthropic()
openai_client = openai.OpenAI()
google_client = genai.Client()

INTER_REQUEST_DELAY = 10   # seconds between requests
RETRY_DELAYS = [30, 30]   # seconds to wait before 1st 2nd, and 3rd retry

def encode_image(file_name, max_side=1024):
    path = os.path.join(RENDER_DIR, file_name)
    img = Image.open(path)
    img.thumbnail((max_side, max_side), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.standard_b64encode(buf.getvalue()).decode("utf-8")


def mime_type(file_name):
    ext = os.path.splitext(file_name)[1].lower()
    return {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".webp": "image/webp",
    }.get(ext, "image/png")


def parse_score(raw):
    match = re.search(r'\d+', raw)
    return int(match.group()) if match else None


def build_example_blocks(shot_df):
    blocks = []
    if shot_df is None:
        return blocks
    for _, row in shot_df.iterrows():
        b64 = encode_image(row["file_name"])
        mt = mime_type(row["file_name"])
        metadata = json.dumps({
            "role": "example",
            "label": int(row["label"]),
            "description": str(row["description"]),
            "comments": str(row["key_comments"]),
        })
        blocks.append({"image": b64, "mime": mt, "text": metadata})
    return blocks


def build_eval_block(row):
    b64 = encode_image(row["file_name"])
    mt = mime_type(row["file_name"])
    metadata = json.dumps({
        "role": "eval",
        "description": str(row["description"]),
        "comments": str(row["key_comments"]),
    })
    return {"image": b64, "mime": mt, "text": metadata}


def call_anthropic(model_id, examples, eval_block):
    content = [{"type": "text", "text": TASK_DESCRIPTION}]
    for ex in examples:
        content.append({
            "type": "image",
            "source": {"type": "base64", "media_type": ex["mime"], "data": ex["image"]},
        })
        content.append({"type": "text", "text": ex["text"]})
    content.append({
        "type": "image",
        "source": {"type": "base64", "media_type": eval_block["mime"], "data": eval_block["image"]},
    })
    content.append({"type": "text", "text": eval_block["text"]})
    response = anthropic_client.messages.create(
        model=model_id,
        max_tokens=16,
        temperature=0,
        messages=[{"role": "user", "content": content}],
    )
    return response.content[0].text.strip()


def call_openai(model_id, examples, eval_block):
    content = [{"type": "text", "text": TASK_DESCRIPTION}]
    for ex in examples:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:{ex['mime']};base64,{ex['image']}"},
        })
        content.append({"type": "text", "text": ex["text"]})
    content.append({
        "type": "image_url",
        "image_url": {"url": f"data:{eval_block['mime']};base64,{eval_block['image']}"},
    })
    content.append({"type": "text", "text": eval_block["text"]})
    response = openai_client.chat.completions.create(
        model=model_id,
        max_completion_tokens=16,
        temperature=0,
        messages=[{"role": "user", "content": content}],
    )
    return response.choices[0].message.content.strip()


def call_google(model_id, examples, eval_block):
    parts = [types.Part.from_text(text=TASK_DESCRIPTION)]
    for ex in examples:
        img_bytes = base64.standard_b64decode(ex["image"])
        parts.append(types.Part.from_bytes(data=img_bytes, mime_type=ex["mime"]))
        parts.append(types.Part.from_text(text=ex["text"]))
    eval_bytes = base64.standard_b64decode(eval_block["image"])
    parts.append(types.Part.from_bytes(data=eval_bytes, mime_type=eval_block["mime"]))
    parts.append(types.Part.from_text(text=eval_block["text"]))
    response = google_client.models.generate_content(
        model=model_id,
        contents=parts,
        config=types.GenerateContentConfig(max_output_tokens=256, temperature=0),
    )
    if response.text is None:
        print(f"  [DEBUG] Empty response. Finish reason: {response.candidates[0].finish_reason if response.candidates else 'no candidates'}")
    return response.text.strip()


PROVIDER_FN = {
    "anthropic": call_anthropic,
    "openai": call_openai,
    "google": call_google,
}


eval_df = pd.read_csv(os.path.join(DATA_DIR, "eval.csv"))
os.makedirs(EVAL_DIR, exist_ok=True)

for model_name, model_cfg in MODELS.items():
    provider = model_cfg["provider"]
    model_id = model_cfg["model_id"]
    call_fn = PROVIDER_FN[provider]

    for setup_name, shot_file in SETUPS.items():
        out_name = f"{model_name}-{setup_name}.csv"
        out_path = os.path.join(EVAL_DIR, out_name)

        # Resume: load existing results and find already-scored file names
        if os.path.exists(out_path):
            existing_df = pd.read_csv(out_path)
            done = set(existing_df.loc[existing_df["score"].notna(), "file_name"])
            results = existing_df.to_dict("records")
            print(f"[RESUME] {out_name}: {len(done)}/{len(eval_df)} already scored, skipping.")
        else:
            done = set()
            results = []

        rows_to_run = eval_df[~eval_df["file_name"].isin(done)]
        if rows_to_run.empty:
            print(f"[SKIP] {out_name} is complete.")
            continue

        print(f"[RUN] Model: {model_name} | Setup: {setup_name} | Remaining: {len(rows_to_run)}")

        shot_df = (
            pd.read_csv(os.path.join(DATA_DIR, shot_file))
            if shot_file is not None
            else None
        )
        examples = build_example_blocks(shot_df)

        for idx, row in rows_to_run.iterrows():
            eval_block = build_eval_block(row)
            score = None
            for attempt, delay in enumerate([0] + RETRY_DELAYS):
                if delay:
                    print(f"  [RETRY {attempt}] Waiting {delay}s before retry for {row['file_name']}...")
                    time.sleep(delay)
                try:
                    raw = call_fn(model_id, examples, eval_block)
                    score = parse_score(raw)
                    if score is not None:
                        break
                    print(f"  [WARN] No integer found for {row['file_name']} (attempt {attempt + 1}): {raw}")
                except Exception as e:
                    print(f"  [ERROR] {row['file_name']} (attempt {attempt + 1}): {e}")
                if attempt == len(RETRY_DELAYS):
                    print(f"  [FAIL] Giving up on {row['file_name']} after {attempt + 1} attempts.")

            results.append({
                "file_name": row["file_name"],
                "label": row["label"],
                "score": score,
            })
            print(f"  {row['file_name']}  label={row['label']}  score={score}")

            # Save after every row so progress is never lost
            pd.DataFrame(results).to_csv(out_path, index=False)
            time.sleep(INTER_REQUEST_DELAY)

        print(f"  -> Saved {out_name} ({len(results)} rows)")

print("Done.")